In [1]:
import numpy as np
import random


In [2]:

# ------------------------
# Robot Environment
# ------------------------
class RobotEnv:
    def __init__(self, size=5):
        self.size = size
        self.start = (0, 0)
        self.goal = (4, 4)
        self.obstacle = (2, 2)
        self.reset()

    def reset(self):
        self.pos = self.start
        return self.pos

    def step(self, action):
        x, y = self.pos

        # Actions: 0=up, 1=down, 2=left, 3=right
        if action == 0: x = max(0, x - 1)
        if action == 1: x = min(self.size - 1, x + 1)
        if action == 2: y = max(0, y - 1)
        if action == 3: y = min(self.size - 1, y + 1)

        self.pos = (x, y)

        if self.pos == self.goal:
            return self.pos, 10, True
        elif self.pos == self.obstacle:
            return self.pos, -5, False
        else:
            return self.pos, -1, False

In [3]:
# ------------------------
# Q-Learning
# ------------------------
env = RobotEnv()
Q = np.zeros((5, 5, 4))

alpha = 0.1
gamma = 0.9
epsilon = 0.3
episodes = 2000
MAX_STEPS = 50   # <-- IMPORTANT FIX

for _ in range(episodes):
    state = env.reset()
    done = False

    for step in range(MAX_STEPS):    # <-- Prevent infinite loops
        x, y = state

        # Epsilon-greedy
        if random.random() < epsilon:
            action = random.randint(0, 3)
        else:
            action = np.argmax(Q[x, y])

        next_state, reward, done = env.step(action)
        nx, ny = next_state

        # Update Q-value
        Q[x, y, action] += alpha * (
            reward + gamma * np.max(Q[nx, ny]) - Q[x, y, action]
        )

        state = next_state

        if done:
            break


In [4]:

# ------------------------
# Test the trained robot
# ------------------------
state = env.reset()
path = [state]
done = False

for step in range(MAX_STEPS):  # <-- avoid infinite loops during testing too
    x, y = state
    action = np.argmax(Q[x, y])
    state, reward, done = env.step(action)
    path.append(state)
    if done:
        break

print("Learned Q-values:")
print(Q)

print("\nRobot’s optimal learned path:")
print(path)


Learned Q-values:
[[[-1.39065581 -0.43406463 -1.3906558  -0.434062  ]
  [-0.43406203  0.62881873 -1.39065596  0.62882   ]
  [ 0.62881989  1.8098     -0.43406233  1.80979972]
  [ 1.67760309  3.122       0.52444704  1.84034486]
  [ 0.10987711  4.36885161  0.69342638 -0.43474214]]

 [[-1.5107297  -0.77906598 -0.9686432   0.62881989]
  [-0.53897724  1.72573206 -0.48972179  1.8098    ]
  [ 0.62881999 -0.87800124  0.62881964  3.122     ]
  [ 1.80979994  4.58        1.80979986  4.57999916]
  [ 1.84049801  6.19999999  2.80847665  3.79827833]]

 [[-1.48598508 -1.55392185 -1.70387057  1.4013081 ]
  [-0.26308965  3.12124879 -0.93280134 -1.6480647 ]
  [ 1.56013324  4.58        1.66110625  4.31778257]
  [ 3.12199676  6.2        -0.87800111  6.19998982]
  [ 4.46233754  8.          4.49233164  6.0809227 ]]

 [[-1.13806425 -1.41486734 -0.96610915  1.92590492]
  [ 0.70369717  1.65839366 -0.54088581  4.57999054]
  [-0.87882755  6.0345546   3.09076449  6.2       ]
  [ 4.57999962  7.99999965  4.57999967  